In [ ]:
import numpy as np
from PIL import Image

# Configuration
INPUT_IMAGE = 'tile_generator_custom/here.png'
OUTPUT_MEM = 'tile_generator_custom/image_data2.mem'
ROWS = 32
COLS = 32

print("... Loading input PNG image")
try:
    # Open the image and ensure it's in RGBA mode
    img = Image.open(INPUT_IMAGE).convert('RGBA')
except FileNotFoundError:
    print(f"Error: Could not find the file at {INPUT_IMAGE}")
    exit()

# Verify dimensions
if img.size != (COLS, ROWS):
    print(f"Warning: Image size is {img.size}, resizing to {COLS}x{ROWS}")
    img = img.resize((COLS, ROWS))

# Convert to a NumPy array of shape (32, 32, 4)
img_array = np.array(img).astype(np.float32)  # Convert to float for precise math

# --- COLOR TWEAKING STEP: Red-Leaning Retro Brown Shift ---
# Target a warmer, redder mahogany/chestnut brown: R=130, G=60, B=30
brown_target = np.array([130, 60, 30])

# 1. Blend 40% of the warm brown target into the original RGB channels
img_array[:, :, :3] = (img_array[:, :, :3] * 0.60) + (brown_target * 0.40)

# 2. Specifically suppress the green channel to prevent "olive" undertones
img_array[:, :, 1] *= 0.85

# 3. Suppress the blue channel to ensure warmth dominates
img_array[:, :, 2] *= 0.75

# 4. Apply global dimming (85% brightness) for that moody, dark arcade bar atmosphere
img_array[:, :, :3] *= 0.85

# Clip values to ensure they stay strictly within valid 0-255 boundaries
img_array[:, :, :3] = np.clip(img_array[:, :, :3], 0, 255)
# -----------------------------------------------------

# Extract individual channels and force them to integers for bit-shifting
# Scale 0-255 down to 2 bits (0-3) by integer dividing by 64
r_2bit = (img_array[:, :, 0] // 64).astype(np.int32)
g_2bit = (img_array[:, :, 1] // 64).astype(np.int32)
b_2bit = (img_array[:, :, 2] // 64).astype(np.int32)

# Alpha channel: standard behavior is 0 = fully transparent, 255 = fully opaque.
# For your 1-bit transparency, let's assume: Transparent = 1, Opaque = 0.
a_1bit = np.where(img_array[:, :, 3] < 128, 1, 0).astype(np.int32)

print("... Mapping pixels to 7-bit format")
# Shift bits into their respective positions:
# A is at bit 6, R at bits 5-4, G at bits 3-2, B at bits 1-0
custom_7bit_array = (a_1bit << 6) | (r_2bit << 4) | (g_2bit << 2) | b_2bit

# Flatten the 2D array into a 1D array of 1024 pixels
flattened_data = custom_7bit_array.flatten()

print(f"... Saving to memory file: {OUTPUT_MEM}")
# Write to a .mem file (text format containing binary strings)
with open(OUTPUT_MEM, 'w') as f:
    for pixel in flattened_data:
        if (pixel >> 6) & 1:
            pixel = 0b1000000
        # Formats the byte as a 7-digit binary string
        f.write(f"{pixel:07b}\n")

print("... File successfully saved!")

... Loading input PNG image
... Mapping pixels to 7-bit format
... Saving to memory file: tile_generator_custom/image_data2.mem
... File successfully saved!
